# Merge sort, step by step

Companion to [W10 §3](../README.md#3-divide-and-conquer-merge-sort). Divide and conquer:
split the list, sort each half the same way, merge the halves. You will:

1. merge two sorted piles by hand;
2. read the whole recursive function;
3. watch eight numbers split down and merge back up;
4. count comparisons against insertion sort;
5. sort a cut list.

**Setup.** Run this cell and move on. It lets the notebook find the drawing helpers
(`viz`) and this week's algorithms (`examples/`), whether it sits in the lecture
folder or is your copy in `MyWork/`.

In [ ]:
import sys
import pathlib

HERE = pathlib.Path.cwd()
SHARED = next(p / "Lecture" / "shared" for p in [HERE, *HERE.parents] if (p / "Lecture" / "shared" / "viz.py").exists())
sys.path[:0] = [str(SHARED), str(SHARED.parent / "Lecture_10" / "examples")]

## Step 1 — Merging two sorted piles

Two piles, each already sorted. Look at the top of both, take the smaller, repeat.
Every comparison places one item, so merging two piles of 4 takes at most 7
comparisons. `counter` adds them up.

In [ ]:
import inspect

import merge_sort

print(inspect.getsource(merge_sort.merge))
counter = [0]
print(merge_sort.merge([3, 27, 38, 43], [9, 10, 51, 82], counter), "in", counter[0], "comparisons")

## Step 2 — The whole function

It calls itself on each half (recursion, Week 08) and merges the two answers. The
**base case** stops it: a list of one item, or none, is already sorted.

In [ ]:
print(inspect.getsource(merge_sort.merge_sort))

## Step 3 — Eight numbers, down and back up

`trace` records every merge with its depth: depth 2 merges single items into pairs,
depth 1 pairs into fours, depth 0 the final list. Read it from the top: the deepest
merges happen first.

In [ ]:
items = [38, 27, 43, 3, 9, 82, 10, 51]
trace = []
result = merge_sort.merge_sort(items, trace=trace)
for depth, merged in trace:
    print("    " * depth, "depth", depth, "merged", merged)
print("result", result)

## Step 4 — Count the comparisons

Insertion sort slides each item back until it fits: on a shuffled list that is about
n²/4 comparisons. Merge sort makes log₂ n levels of merges, each touching every item
once: at most n log₂ n. Both sort the same lists below; watch the gap grow.

In [ ]:
import random

from viz import chart

sizes = list(range(100, 2001, 100))
slow, fast = [], []
for n in sizes:
    data = random.Random(n).sample(range(100_000), n)
    a, b = [0], [0]
    merge_sort.insertion_sort(data, a)
    merge_sort.merge_sort(data, b)
    slow.append(a[0])
    fast.append(b[0])
print(f"n = {sizes[-1]}: insertion {slow[-1]:,}, merge {fast[-1]:,} comparisons")
chart({"insertion sort": slow, "merge sort": fast}, "comparisons as the list grows", "n = 100 ... 2,000")

## Step 5 — A cut list

Twenty timber boards to cut, in the order the model produced them (left). Sorted by
length (right), the saw is set once per length and the longest come first off the
stock. Each bar is one board.

In [ ]:
from viz import draw_points, row

rnd = random.Random(4)
boards = [round(rnd.uniform(0.4, 2.4), 2) for _ in range(20)]
cut_list = merge_sort.merge_sort(boards)[::-1]
print("cut list (m):", cut_list)


def bars(lengths, title):
    gap = 0.12                  # metres between bars on the drawing
    return draw_points([], title, lines=[((0, -gap * k), (length, -gap * k)) for k, length in enumerate(lengths)],
                       bounds=[[0, 0], [2.5, -gap * len(lengths)]], size=(380, 320),
                       line_color="#e8590c", line_width=7)


row(bars(boards, "as modelled"), bars(cut_list, "sorted, longest first"))

## Your turn

1. Count insertion sort's comparisons on a list that is already sorted, and on one
   sorted backwards. Which is its best case, which its worst? Does merge sort care?
2. Sort room names (`["stair", "atrium", "lobby", "core"]`): what does `<` mean for text?
3. Change the base case to `if len(items) == 1`. Call `merge_sort([])`: what happens,
   and why?